# Planner Agent

LangGraph-based agent that breaks a travel request into actionable tasks (airfare, hotel, car rental).

In [1]:
import asyncio
import logging
import os
from collections.abc import AsyncIterable
from typing import Any, Literal

import uvicorn
import weave
from dotenv import load_dotenv
from langchain_core.messages import AIMessage
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field

import prompts
from common import BaseAgent, TaskList, build_a2a_app, init_api_key

load_dotenv()
init_api_key()
logger = logging.getLogger(__name__)

WANDB_PROJECT = os.getenv('WANDB_WORKSPACE')
weave_client = weave.init(WANDB_PROJECT)
print(f'Weave initialized: {WANDB_PROJECT}')

/Users/paul/.cache/uv/archive-v0/pqxv7ELpHFQOMM0CmHnVk/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
/Users/paul/Documents/code/a2a-examples/.venv/lib/python3.13/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils
weave: Logged in as Weights & Biases user: paulbruffett.
weave: View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave


Weave initialized: pbruffett/a2a-lab


In [2]:
class ResponseFormat(BaseModel):
    """Respond to the user in this format."""
    status: Literal['input_required', 'completed', 'error', 'pending'] = 'input_required'
    question: str = Field(description='Input needed from the user')
    content: TaskList = Field(description='List of tasks when the plan is generated')


class LangGraphPlannerAgent(BaseAgent):
    def __init__(self):
        super().__init__(agent_name='PlannerAgent', description='Breakdown user request into tasks', content_types=['text', 'text/plain'])
        self.model = ChatAnthropic(model='claude-haiku-4-5', temperature=0.0)
        self.graph = create_react_agent(
            self.model, checkpointer=MemorySaver(),
            prompt=prompts.PLANNER_COT_INSTRUCTIONS,
            response_format=ResponseFormat, tools=[],
        )

    @weave.op()
    async def stream(self, query, session_id, task_id) -> AsyncIterable[dict[str, Any]]:
        logger.info('[planner.stream] REQUEST session=%s task=%s query=%r', session_id, task_id, query)
        config = {'configurable': {'thread_id': session_id}}
        for item in self.graph.stream({'messages': [('user', query)]}, config, stream_mode='values'):
            message = item['messages'][-1]
            if isinstance(message, AIMessage):
                content = message.content if isinstance(message.content, str) else str(message.content)
                logger.info('[planner.stream] YIELD interim content=%r', content)
                yield {'response_type': 'text', 'is_task_complete': False, 'require_user_input': False, 'content': content}
        final = self._get_response(config)
        logger.info('[planner.stream] YIELD final=%s', final)
        yield final

    @weave.op()
    def _get_response(self, config):
        resp = self.graph.get_state(config).values.get('structured_response')
        if isinstance(resp, ResponseFormat):
            if resp.status in ('completed', 'pending'):
                return {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': resp.content.model_dump()}
            return {'response_type': 'text', 'is_task_complete': False, 'require_user_input': True, 'content': resp.question}
        return {'is_task_complete': False, 'require_user_input': True, 'content': 'Unable to process request. Please try again.'}

In [ ]:
app = build_a2a_app(LangGraphPlannerAgent(), 'agent_cards/planner_agent.json')

if __name__ == "__main__":
    config = uvicorn.Config(app=app, host='localhost', port=10102)
    server = uvicorn.Server(config)
    await server.serve()

/var/folders/ch/wjmp4fss3gvbg92jslrpt8_m0000gn/T/ipykernel_6549/1955205168.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  self.graph = create_react_agent(
INFO:     Started server process [6549]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10102 (Press CTRL+C to quit)


INFO:     ::1:55379 - "POST / HTTP/1.1" 200 OK


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d949e-4bea-7f55-a045-0e73348ebf03
Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


INFO:     ::1:55387 - "POST / HTTP/1.1" 200 OK


Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]
weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d949e-8061-702c-9532-e23d8cbf82e1
Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


INFO:     ::1:55396 - "POST / HTTP/1.1" 200 OK


Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]
weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d949f-2269-7f63-99b8-e6e1f720df11
Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]
